In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.features.feature_engineer.apm_features import _detect_star_players
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.utils.helpers import *
from src.utils.dataScraper import *
from live import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'BOS': ['Jayson Tatum']}

Out Players:
{}
Note: PHI (76ers) has 3 confirmed players - lineup will still be updated
Note: BOS (Celtics) has 4 confirmed players - lineup will still be updated
Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 2 teams with confirmed lineups
Updated 1 teams with questionable players


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
p25 = pd.read_csv('data/raw/playoff_stats/P25.csv').sort_values(by='GAME_DATE')
s25 = pd.concat([s25, p25])
s25 = _detect_star_players(s25)

s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
s26 = pd.concat([s26, p26])
s26 = _detect_star_players(s26)

base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,TOP_PLAYER,SECOND_TOP_PLAYER,THIRD_TOP_PLAYER,IS_TOP_STAR,IS_TOP_1_STAR,ACTIVE_STARS_COUNT,TOP_STAR_ACTIVE,name
27650,NaN,NaN,NaN,2025-26,1627751,Jakob Poeltl,Jakob,1610612761,TOR,Toronto Raptors,42500136,2026-05-01T00:00:00,TOR vs. CLE,W,21.516667,1,3,0.333,0,0,0.0,0,0,0.00,2,2,4,1,0,2,1,1,3,0,2,0,17.3,0,0,13.0,1,21:31,1,112.0,109.1,109.1,105.0,106.7,106.7,7.1,2.4,2.4,0.063,0.00,25.0,0.100,0.095,0.098,0.0,0.0,0.333,0.333,0.061,0.063,98.78,99.27,82.73,99.27,0.048,44,1.0,3.0,C,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40,87,0.460,13,36,0.361,19,23,0.826,10,28,38,27,16.0,13,9,4,25,21,112,2.0,108.6,110.9,105.9,107.8,2.7,3.0,0.675,1.69,19.0,0.313,0.536,0.433,0.158,0.534,0.577,93.7,91.92,76.60,101,0.533,1610612739,CLE,Cleveland Cavaliers,40,93,0.43,11,41,0.268,19,27,0.704,19,33,52,23,18.0,8,4,9,21,25,110,-2.0,105.9,107.8,108.6,110.9,-2.7,-3.0,0.575,1.28,15.2,0.464,0.688,0.567,0.176,0.489,0.524,93.7,91.92,76.60,102,0.467,1,C,30.0,NaN,NaN,NaN,NaN,1,0.092951,0.046476,0.185902,1,0,Scottie Barnes,Brandon Ingram,RJ Barrett,0,0,2,1,Jakob Poeltl
27651,NaN,NaN,NaN,2025-26,1631288,Jamal Cain,Jamal,1610612753,ORL,Orlando Magic,42500106,2026-05-01T00:00:00,ORL vs. DET,L,20.150000,1,3,0.333,1,2,0.5,0,0,0.00,1,3,4,2,1,1,1,1,4,2,3,-7,15.8,0,0,14.0,1,20:09,1,92.7,92.1,92.1,113.3,105.0,105.0,-20.6,-12.9,-12.9,0.167,2.00,33.3,0.048,0.167,0.103,16.7,16.7,0.500,0.500,0.091,0.096,89.14,92.90,77.42,92.90,0.045,38,1.0,3.0,F,4.14,1.45,2.0,3.0,5.0,15.0,0.0,0.0,10.0,0.0,1.0,0.0,1.0,2.0,0.5,1.0,2.0,0.5,27,78,0.346,9,36,0.250,16,21,0.762,8,30,38,20,11.0,6,4,8,22,21,79,-14.0,87.5,89.8,105.2,105.7,-17.6,-15.9,0.741,1.82,16.5,0.241,0.685,0.463,0.125,0.404,0.453,89.3,88.00,73.33,88,0.403,1610612765,DET,Detroit Pistons,32,80,0.40,9,27,0.333,20,26,0.769,14,38,52,16,11.0,5,8,4,21,22,93,14.0,105.2,105.7,87.5,89.8,17.6,15.9,0.500,1.45,13.4,0.315,0.759,0.537,0.125,0.456,0.509,89.3,88.00,73.33,88,0.597,1,SF,26.0,NaN,NaN,NaN,NaN,1,0.148883,0.099256,0.198511,1,3,Paolo Banchero,Franz Wagner,Desmond Bane,0,0,2,1,Jamal Cain
27652,NaN,NaN,NaN,2025-26,1631222,Jake LaRavia,Jake,1610612747,LAL,Los Angeles 

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_odds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_odds = pd.json_normalize(data)

print("Loaded:", file.name)
team_odds.head()

Loaded: NBA_20260502_105937.json


,home_team,away_team,commence_time,bookmakers
0,Boston Celtics,Philadelphia 76ers,2026-05-02 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Detroit Pistons,Orlando Magic,2026-05-03 19:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,Cleveland Cavaliers,Toronto Raptors,2026-05-03 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
3,San Antonio Spurs,Minnesota Timberwolves,2026-05-05 01:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
4,Oklahoma City Thunder,Los Angeles Lakers,2026-05-06 01:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = s26
ast_df = s26
reb_df = s26
min_df = s26


#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs = lines_dfs[lines_dfs['COMMENCE_TIME'] == current_date]
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us = lines_us[lines_us['COMMENCE_TIME'] == current_date]
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-05-02 10:58:55
US latest pull: 2026-05-02 10:59:37


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Tyrese Maxey,Over,0.5,-137,2026-05-02,2026-05-02T17:57:57Z,2026-05-02 10:58:55
1,PrizePicks,player_points,Jaylen Brown,Over,26.5,-137,2026-05-02,2026-05-02T17:57:57Z,2026-05-02 10:58:55
2,PrizePicks,player_points,Jaylen Brown,Under,26.5,-137,2026-05-02,2026-05-02T17:57:57Z,2026-05-02 10:58:55
3,PrizePicks,player_points,Joel Embiid,Over,25.5,-137,2026-05-02,2026-05-02T17:57:57Z,2026-05-02 10:58:55
4,PrizePicks,player_points,Joel Embiid,Under,25.5,-137,2026-05-02,2026-05-02T17:57:57Z,2026-05-02 10:58:55


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb_2026-01-02.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb_2026-01-01.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb_2026-01-01.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb_2026-01-01.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
pts_preds.head(10)

[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)
[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Tyrese Maxey,PTS,31.49,41.26,45.06,0.3627,0.5938,0.9498,11.42,24.50,42.80,"[0.6045808626903848, 0.5237191650853891, 0.823..."
1,Jaylen Brown,PTS,28.78,37.53,42.25,0.4134,0.6387,0.9613,11.90,23.97,40.61,"[0.7862665443585375, 0.822461712989223, 0.7969..."
2,Joel Embiid,PTS,22.65,29.02,36.26,0.4028,0.6329,0.9818,9.12,18.37,35.60,"[1.0303058430407943, 1.0252029047415634, 0.744..."
3,Jayson Tatum,PTS,24.53,28.95,38.09,0.3822,0.5904,0.9235,9.38,17.09,35.17,"[0.5208898534997287, 0.5364705882352941, 0.710..."
4,Paul George,PTS,23.04,28.55,36.51,0.2790,0.4915,0.7551,6.43,14.03,27.57,"[0.7149404216315307, 0.5468937395058767, 1.283..."
5,Payton Pritchard,PTS,23.18,31.00,36.68,0.2443,0.4771,0.7377,5.66,14.79,27.06,"[1.041817392562581, 0.7520143240823636, 0.6261..."
6,Derrick White,PTS,27.67,36.33,41.10,0.1731,0.4422,0.7451,4.79,16.06,30.63,"[0.3622459247333467, 0.292141396435875, 0.1962..."
7,VJ Edgecombe,PTS,31.44,39.55,43.30,0.1926,0.4378,0.7398,6.06,17.32,32.03,"[0.4229017566688354, 0.4346614655893007, 0.699..."
8,Neemias Queta,PTS,15.20,22.30,30.26,0.1485,0.4461,0.7187,2.26,9.95,21.75,"[0.4347341433507971, 0.1812907904278462, 0.591..."
9,Quentin Grimes,PTS,18.05,24.14,31.36,0.1854,0.4340,0.7333,3.35,10.48,23.00,"[0.2803738317757009, 0.3481012658227848, 0.519..."


In [23]:
# from live import adjust_predictions

# # Build contexts dict once (using the notebook's get_game_context)
# game_contexts = {
#     name: get_game_context(base_df, name, team_odds)
#     for name in pts_preds["PLAYER_NAME"]
# }

# # Adjust the model's Q50 predictions with scenario signals
# pts_preds = adjust_predictions(pts_preds, base_df, game_contexts)
# ast_preds = adjust_predictions(ast_preds, base_df, game_contexts)
# reb_preds = adjust_predictions(reb_preds, base_df, game_contexts)
# ast_preds

### Get Line Probabilities

In [8]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
26,VJ Edgecombe,PTS,11.5,31.44,39.55,43.30,6.06,17.32,32.03,0.653,0.347
15,Jayson Tatum,REB,10.5,24.53,28.95,38.09,4.13,7.66,15.44,0.450,0.550
3,Jaylen Brown,AST,4.5,28.78,37.53,42.25,1.40,4.33,8.04,0.353,0.647
25,Derrick White,PTS,11.5,27.67,36.33,41.10,4.79,16.06,30.63,0.483,0.517
9,Jaylen Brown,REB,6.0,28.78,37.53,42.25,2.55,6.50,11.37,0.477,0.523
23,Paul George,PTS,15.5,23.04,28.55,36.51,6.43,14.03,27.57,0.527,0.472
0,Tyrese Maxey,AST,6.5,31.49,41.26,45.06,2.49,6.26,11.07,0.414,0.586
2,Joel Embiid,AST,5.0,22.65,29.02,36.26,1.43,3.92,8.60,0.518,0.482
30,Sam Hauser,PTS,6.5,16.79,23.21,31.56,2.00,9.25,22.44,0.713,0.287
17,Jordan Walsh,REB,2.5,15.37,19.38,24.60,1.36,3.80,8.29,0.778,0.222


In [18]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY_x,LINE_BOOKMAKER_x,OPPONENT_x,TEAM_SPREAD_x,GAME_TOTAL_x,OPP_DEF_RATING_x,OPP_RANK_DEF_RATING_x,OPP_PACE_x,OPP_PACE_RANK_x,ODDS_OVER_x,ODDS_UNDER_x,IMP_PROB_OVER_x,IMP_PROB_UNDER_x,AVG_STAT_L10_x,MED_STAT_L10_x,STD_STAT_L10_x,EDGE_x,MED_EDGE_x,Z_SCORE_x,PROB_OVER_x,PROB_UNDER_x,EV_OVER_x,EV_UNDER_x,OVER_RATE_L5_x,OVER_RATE_L10_x,OVER_RATE_L15_x,OVER_RATE_SEASON_x,AVG_MIN_L10_x,STD_MIN_L10_x,AVG_USG_L10_x,STD_USG_L10_x,AVG_STAT_VS_MATCHUP_x,MATCHUP_GAMES_x,_merge,CATEGORY_y,LINE_BOOKMAKER_y,OPPONENT_y,TEAM_SPREAD_y,GAME_TOTAL_y,OPP_DEF_RATING_y,OPP_RANK_DEF_RATING_y,OPP_PACE_y,OPP_PACE_RANK_y,ODDS_OVER_y,ODDS_UNDER_y,IMP_PROB_OVER_y,IMP_PROB_UNDER_y,AVG_STAT_L10_y,MED_STAT_L10_y,STD_STAT_L10_y,EDGE_y,MED_EDGE_y,Z_SCORE_y,PROB_OVER_y,PROB_UNDER_y,EV_OVER_y,EV_UNDER_y,OVER_RATE_L5_y,OVER_RATE_L10_y,OVER_RATE_L15_y,OVER_RATE_SEASON_y,AVG_MIN_L10_y,STD_MIN_L10_y,AVG_USG_L10_y,STD_USG_L10_y,AVG_STAT_VS_MATCHUP_y,MATCHUP_GAMES_y
1,Jayson Tatum,AST,5.5,24.53,28.95,38.09,1.90,4.22,9.08,0.472,0.528,player_assists,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-132.0,110.0,0.569,0.476,6.9,7.0,2.56,1.4,1.5,-0.547,0.708,0.292,24.44,-38.68,0.6,0.7,0.67,0.53,36.19,4.56,0.27,0.04,6.80,10.0,both,AST,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-132.0,110.0,0.569,0.476,6.9,7.0,2.56,1.4,1.5,-0.547,0.708,0.292,24.44,-38.68,0.6,0.7,0.67,0.53,36.19,4.56,0.27,0.04,6.80,10.0
12,Derrick White,REB,3.5,27.67,36.33,41.10,1.53,4.11,9.61,0.524,0.476,player_rebounds,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,100.0,100.0,0.500,0.500,3.0,2.5,1.83,-0.5,-1.0,0.273,0.392,0.608,-21.60,21.60,0.4,0.3,0.40,0.64,33.06,6.46,0.15,0.06,3.71,14.0,both,REB,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,100.0,100.0,0.500,0.500,3.0,2.5,1.83,-0.5,-1.0,0.273,0.392,0.608,-21.60,21.60,0.4,0.3,0.40,0.64,33.06,6.46,0.15,0.06,3.71,14.0
15,Jayson Tatum,REB,10.5,24.53,28.95,38.09,4.13,7.66,15.44,0.450,0.550,player_rebounds,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,108.0,-129.0,0.481,0.563,10.6,11.0,3.78,0.1,0.5,-0.026,0.510,0.490,6.08,-13.02,0.6,0.7,0.73,0.36,36.19,4.56,0.27,0.04,10.40,10.0,both,REB,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,108.0,-129.0,0.481,0.563,10.6,11.0,3.78,0.1,0.5,-0.026,0.510,0.490,6.08,-13.02,0.6,0.7,0.73,0.36,36.19,4.56,0.27,0.04,10.40,10.0
19,Tyrese Maxey,PTS,24.5,31.49,41.26,45.06,11.42,24.50,42.80,0.631,0.369,player_points,Underdog,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-104.0,-116.0,0.510,0.537,25.0,24.5,5.46,0.5,0.0,-0.092,0.537,0.463,5.33,-13.79,0.8,0.5,0.47,0.67,37.96,4.71,0.27,0.05,27.77,13.0,both,PTS,Underdog,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-104.0,-116.0,0.510,0.537,25.0,24.5,5.46,0.5,0.0,-0.092,0.537,0.463,5.33,-13.79,0.8,0.5,0.47,0.67,37.96,4.71,0.27,0.05,27.77,13.0
20,Jaylen Brown,PTS,26.5,28.78,37.53,42.25,11.90,23.97,40.61,0.487,0.513,player_points,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,105.0,-114.0,0.488,0.533,25.7,25.5,5.83,-0.8,-1.0,0.137,0.446,0.554,-8.57,4.00,0.2,0.2,0.47,0.44,34.64,5.61,0.33,0.04,24.54,13.0,both,PTS,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,105.0,-114.0,0.488,0.533,25.7,25.5,5.83,-0.8,-1.0,0.137,0.446,0.554,-8.57,4.00,0.2,0.2,0.47,0.44,34.64,5.61,0.33,0.04,24.54,13.0
22,Jayson Tatum,PTS,23.5,24.53,28.95,38.09,9.38,17.09,35.17,0.304,0.696,player_points,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-103.0,100.0,0.507,0.500,23.3,23.5,3.50,-0.2,0.0,0.057,0.477,0.523,-5.99,4.60,0.6,0.5,0.53,0.59,36.19,4.56,0.27,0.04,25.70,10.0,both,PTS,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-103.0,100.0,0.507,0.500,23.3,23.5,3.50,-0.2,0.0,0.057,0.477,0.523,-5.99,4.60,0.6,0.5,0.53,0.59,36.19,4.56,0.27,0.04,25.70,10.0
23,Paul George,PTS,15.5,23.04,28.55,36.51,6.43,14.03,27.57,0.527,0.472,player_points,Underdog,Boston Celtics,7.5,204.5,111.7

In [19]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY_x,LINE_BOOKMAKER_x,OPPONENT_x,TEAM_SPREAD_x,GAME_TOTAL_x,OPP_DEF_RATING_x,OPP_RANK_DEF_RATING_x,OPP_PACE_x,OPP_PACE_RANK_x,ODDS_OVER_x,ODDS_UNDER_x,IMP_PROB_OVER_x,IMP_PROB_UNDER_x,AVG_STAT_L10_x,MED_STAT_L10_x,STD_STAT_L10_x,EDGE_x,MED_EDGE_x,Z_SCORE_x,PROB_OVER_x,PROB_UNDER_x,EV_OVER_x,EV_UNDER_x,OVER_RATE_L5_x,OVER_RATE_L10_x,OVER_RATE_L15_x,OVER_RATE_SEASON_x,AVG_MIN_L10_x,STD_MIN_L10_x,AVG_USG_L10_x,STD_USG_L10_x,AVG_STAT_VS_MATCHUP_x,MATCHUP_GAMES_x,_merge,CATEGORY_y,LINE_BOOKMAKER_y,OPPONENT_y,TEAM_SPREAD_y,GAME_TOTAL_y,OPP_DEF_RATING_y,OPP_RANK_DEF_RATING_y,OPP_PACE_y,OPP_PACE_RANK_y,ODDS_OVER_y,ODDS_UNDER_y,IMP_PROB_OVER_y,IMP_PROB_UNDER_y,AVG_STAT_L10_y,MED_STAT_L10_y,STD_STAT_L10_y,EDGE_y,MED_EDGE_y,Z_SCORE_y,PROB_OVER_y,PROB_UNDER_y,EV_OVER_y,EV_UNDER_y,OVER_RATE_L5_y,OVER_RATE_L10_y,OVER_RATE_L15_y,OVER_RATE_SEASON_y,AVG_MIN_L10_y,STD_MIN_L10_y,AVG_USG_L10_y,STD_USG_L10_y,AVG_STAT_VS_MATCHUP_y,MATCHUP_GAMES_y
1,Jayson Tatum,AST,5.5,24.53,28.95,38.09,1.90,4.22,9.08,0.472,0.528,player_assists,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-132.0,110.0,0.569,0.476,6.9,7.0,2.56,1.4,1.5,-0.547,0.708,0.292,24.44,-38.68,0.6,0.7,0.67,0.53,36.19,4.56,0.27,0.04,6.80,10.0,both,AST,PrizePicks,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-132.0,110.0,0.569,0.476,6.9,7.0,2.56,1.4,1.5,-0.547,0.708,0.292,24.44,-38.68,0.6,0.7,0.67,0.53,36.19,4.56,0.27,0.04,6.80,10.0
12,Derrick White,REB,3.5,27.67,36.33,41.10,1.53,4.11,9.61,0.524,0.476,player_rebounds,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,100.0,100.0,0.500,0.500,3.0,2.5,1.83,-0.5,-1.0,0.273,0.392,0.608,-21.60,21.60,0.4,0.3,0.40,0.64,33.06,6.46,0.15,0.06,3.71,14.0,both,REB,PrizePicks,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,100.0,100.0,0.500,0.500,3.0,2.5,1.83,-0.5,-1.0,0.273,0.392,0.608,-21.60,21.60,0.4,0.3,0.40,0.64,33.06,6.46,0.15,0.06,3.71,14.0
23,Paul George,PTS,15.5,23.04,28.55,36.51,6.43,14.03,27.57,0.527,0.472,player_points,Underdog,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-111.0,100.0,0.526,0.500,16.4,16.5,4.62,0.9,1.0,-0.195,0.577,0.423,9.68,-15.40,1.0,0.8,0.87,0.51,33.51,7.34,0.20,0.03,17.25,8.0,both,PTS,PrizePicks,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-111.0,100.0,0.526,0.500,16.4,16.5,4.62,0.9,1.0,-0.195,0.577,0.423,9.68,-15.40,1.0,0.8,0.87,0.51,33.51,7.34,0.20,0.03,17.25,8.0
22,Jayson Tatum,PTS,23.5,24.53,28.95,38.09,9.38,17.09,35.17,0.304,0.696,player_points,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-103.0,100.0,0.507,0.500,23.3,23.5,3.50,-0.2,0.0,0.057,0.477,0.523,-5.99,4.60,0.6,0.5,0.53,0.59,36.19,4.56,0.27,0.04,25.70,10.0,both,PTS,PrizePicks,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-103.0,100.0,0.507,0.500,23.3,23.5,3.50,-0.2,0.0,0.057,0.477,0.523,-5.99,4.60,0.6,0.5,0.53,0.59,36.19,4.56,0.27,0.04,25.70,10.0
19,Tyrese Maxey,PTS,24.5,31.49,41.26,45.06,11.42,24.50,42.80,0.631,0.369,player_points,Underdog,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-104.0,-116.0,0.510,0.537,25.0,24.5,5.46,0.5,0.0,-0.092,0.537,0.463,5.33,-13.79,0.8,0.5,0.47,0.67,37.96,4.71,0.27,0.05,27.77,13.0,both,PTS,PrizePicks,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-137.0,-137.0,0.578,0.578,25.0,24.5,5.46,24.5,24.0,-4.487,1.000,0.000,72.99,-100.00,1.0,1.0,1.00,1.00,37.96,4.71,0.27,0.05,27.77,13.0


In [20]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY_x,LINE_BOOKMAKER_x,OPPONENT_x,TEAM_SPREAD_x,GAME_TOTAL_x,OPP_DEF_RATING_x,OPP_RANK_DEF_RATING_x,OPP_PACE_x,OPP_PACE_RANK_x,ODDS_OVER_x,ODDS_UNDER_x,IMP_PROB_OVER_x,IMP_PROB_UNDER_x,AVG_STAT_L10_x,MED_STAT_L10_x,STD_STAT_L10_x,EDGE_x,MED_EDGE_x,Z_SCORE_x,PROB_OVER_x,PROB_UNDER_x,EV_OVER_x,EV_UNDER_x,OVER_RATE_L5_x,OVER_RATE_L10_x,OVER_RATE_L15_x,OVER_RATE_SEASON_x,AVG_MIN_L10_x,STD_MIN_L10_x,AVG_USG_L10_x,STD_USG_L10_x,AVG_STAT_VS_MATCHUP_x,MATCHUP_GAMES_x,_merge,CATEGORY_y,LINE_BOOKMAKER_y,OPPONENT_y,TEAM_SPREAD_y,GAME_TOTAL_y,OPP_DEF_RATING_y,OPP_RANK_DEF_RATING_y,OPP_PACE_y,OPP_PACE_RANK_y,ODDS_OVER_y,ODDS_UNDER_y,IMP_PROB_OVER_y,IMP_PROB_UNDER_y,AVG_STAT_L10_y,MED_STAT_L10_y,STD_STAT_L10_y,EDGE_y,MED_EDGE_y,Z_SCORE_y,PROB_OVER_y,PROB_UNDER_y,EV_OVER_y,EV_UNDER_y,OVER_RATE_L5_y,OVER_RATE_L10_y,OVER_RATE_L15_y,OVER_RATE_SEASON_y,AVG_MIN_L10_y,STD_MIN_L10_y,AVG_USG_L10_y,STD_USG_L10_y,AVG_STAT_VS_MATCHUP_y,MATCHUP_GAMES_y
28,Quentin Grimes,PTS,6.5,18.05,24.14,31.36,3.35,10.48,23.00,0.672,0.328,player_points,Underdog,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-114.0,106.0,0.533,0.485,9.6,6.5,7.31,3.1,0.0,-0.424,0.664,0.336,24.65,-30.78,0.4,0.5,0.60,0.76,22.42,4.14,0.17,0.07,10.31,13.0,both,PTS,Betr DFS,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-114.0,106.0,0.533,0.485,9.6,6.5,7.31,3.1,0.0,-0.424,0.664,0.336,24.65,-30.78,0.4,0.5,0.60,0.76,22.42,4.14,0.17,0.07,10.31,13.0
23,Paul George,PTS,15.5,23.04,28.55,36.51,6.43,14.03,27.57,0.527,0.472,player_points,Underdog,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-111.0,100.0,0.526,0.500,16.4,16.5,4.62,0.9,1.0,-0.195,0.577,0.423,9.68,-15.40,1.0,0.8,0.87,0.51,33.51,7.34,0.20,0.03,17.25,8.0,both,PTS,Betr DFS,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-111.0,100.0,0.526,0.500,16.4,16.5,4.62,0.9,1.0,-0.195,0.577,0.423,9.68,-15.40,1.0,0.8,0.87,0.51,33.51,7.34,0.20,0.03,17.25,8.0
15,Jayson Tatum,REB,10.5,24.53,28.95,38.09,4.13,7.66,15.44,0.450,0.550,player_rebounds,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,108.0,-129.0,0.481,0.563,10.6,11.0,3.78,0.1,0.5,-0.026,0.510,0.490,6.08,-13.02,0.6,0.7,0.73,0.36,36.19,4.56,0.27,0.04,10.40,10.0,both,REB,Betr DFS,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,108.0,-129.0,0.481,0.563,10.6,11.0,3.78,0.1,0.5,-0.026,0.510,0.490,6.08,-13.02,0.6,0.7,0.73,0.36,36.19,4.56,0.27,0.04,10.40,10.0
27,Neemias Queta,PTS,7.5,15.20,22.30,30.26,2.26,9.95,21.75,0.680,0.320,player_points,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-112.0,-105.0,0.528,0.512,9.5,8.5,4.01,2.0,1.0,-0.499,0.691,0.309,30.80,-39.67,0.6,0.7,0.73,0.52,21.60,7.00,0.14,0.03,8.77,13.0,both,PTS,Betr DFS,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-112.0,-105.0,0.528,0.512,9.5,8.5,4.01,2.0,1.0,-0.499,0.691,0.309,30.80,-39.67,0.6,0.7,0.73,0.52,21.60,7.00,0.14,0.03,8.77,13.0
20,Jaylen Brown,PTS,26.5,28.78,37.53,42.25,11.90,23.97,40.61,0.487,0.513,player_points,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,105.0,-114.0,0.488,0.533,25.7,25.5,5.83,-0.8,-1.0,0.137,0.446,0.554,-8.57,4.00,0.2,0.2,0.47,0.44,34.64,5.61,0.33,0.04,24.54,13.0,both,PTS,Betr DFS,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,105.0,-114.0,0.488,0.533,25.7,25.5,5.83,-0.8,-1.0,0.137,0.446,0.554,-8.57,4.00,0.2,0.2,0.47,0.44,34.64,5.61,0.33,0.04,24.54,13.0


In [21]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY_x,LINE_BOOKMAKER_x,OPPONENT_x,TEAM_SPREAD_x,GAME_TOTAL_x,OPP_DEF_RATING_x,OPP_RANK_DEF_RATING_x,OPP_PACE_x,OPP_PACE_RANK_x,ODDS_OVER_x,ODDS_UNDER_x,IMP_PROB_OVER_x,IMP_PROB_UNDER_x,AVG_STAT_L10_x,MED_STAT_L10_x,STD_STAT_L10_x,EDGE_x,MED_EDGE_x,Z_SCORE_x,PROB_OVER_x,PROB_UNDER_x,EV_OVER_x,EV_UNDER_x,OVER_RATE_L5_x,OVER_RATE_L10_x,OVER_RATE_L15_x,OVER_RATE_SEASON_x,AVG_MIN_L10_x,STD_MIN_L10_x,AVG_USG_L10_x,STD_USG_L10_x,AVG_STAT_VS_MATCHUP_x,MATCHUP_GAMES_x,_merge,CATEGORY_y,LINE_BOOKMAKER_y,OPPONENT_y,TEAM_SPREAD_y,GAME_TOTAL_y,OPP_DEF_RATING_y,OPP_RANK_DEF_RATING_y,OPP_PACE_y,OPP_PACE_RANK_y,ODDS_OVER_y,ODDS_UNDER_y,IMP_PROB_OVER_y,IMP_PROB_UNDER_y,AVG_STAT_L10_y,MED_STAT_L10_y,STD_STAT_L10_y,EDGE_y,MED_EDGE_y,Z_SCORE_y,PROB_OVER_y,PROB_UNDER_y,EV_OVER_y,EV_UNDER_y,OVER_RATE_L5_y,OVER_RATE_L10_y,OVER_RATE_L15_y,OVER_RATE_SEASON_y,AVG_MIN_L10_y,STD_MIN_L10_y,AVG_USG_L10_y,STD_USG_L10_y,AVG_STAT_VS_MATCHUP_y,MATCHUP_GAMES_y
20,Jaylen Brown,PTS,26.5,28.78,37.53,42.25,11.90,23.97,40.61,0.487,0.513,player_points,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,105.0,-114.0,0.488,0.533,25.7,25.5,5.83,-0.8,-1.0,0.137,0.446,0.554,-8.57,4.00,0.2,0.2,0.47,0.44,34.64,5.61,0.33,0.04,24.54,13.0,both,PTS,DraftKings Pick6,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,105.0,-114.0,0.488,0.533,25.7,25.5,5.83,-0.8,-1.0,0.137,0.446,0.554,-8.57,4.00,0.2,0.2,0.47,0.44,34.64,5.61,0.33,0.04,24.54,13.0
22,Jayson Tatum,PTS,23.5,24.53,28.95,38.09,9.38,17.09,35.17,0.304,0.696,player_points,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-103.0,100.0,0.507,0.500,23.3,23.5,3.50,-0.2,0.0,0.057,0.477,0.523,-5.99,4.60,0.6,0.5,0.53,0.59,36.19,4.56,0.27,0.04,25.70,10.0,both,PTS,DraftKings Pick6,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-103.0,100.0,0.507,0.500,23.3,23.5,3.50,-0.2,0.0,0.057,0.477,0.523,-5.99,4.60,0.6,0.5,0.53,0.59,36.19,4.56,0.27,0.04,25.70,10.0
28,Quentin Grimes,PTS,6.5,18.05,24.14,31.36,3.35,10.48,23.00,0.672,0.328,player_points,Underdog,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-114.0,106.0,0.533,0.485,9.6,6.5,7.31,3.1,0.0,-0.424,0.664,0.336,24.65,-30.78,0.4,0.5,0.60,0.76,22.42,4.14,0.17,0.07,10.31,13.0,both,PTS,DraftKings Pick6,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-114.0,106.0,0.533,0.485,9.6,6.5,7.31,3.1,0.0,-0.424,0.664,0.336,24.65,-30.78,0.4,0.5,0.60,0.76,22.42,4.14,0.17,0.07,10.31,13.0
27,Neemias Queta,PTS,7.5,15.20,22.30,30.26,2.26,9.95,21.75,0.680,0.320,player_points,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-112.0,-105.0,0.528,0.512,9.5,8.5,4.01,2.0,1.0,-0.499,0.691,0.309,30.80,-39.67,0.6,0.7,0.73,0.52,21.60,7.00,0.14,0.03,8.77,13.0,both,PTS,DraftKings Pick6,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-112.0,-105.0,0.528,0.512,9.5,8.5,4.01,2.0,1.0,-0.499,0.691,0.309,30.80,-39.67,0.6,0.7,0.73,0.52,21.60,7.00,0.14,0.03,8.77,13.0
23,Paul George,PTS,15.5,23.04,28.55,36.51,6.43,14.03,27.57,0.527,0.472,player_points,Underdog,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-111.0,100.0,0.526,0.500,16.4,16.5,4.62,0.9,1.0,-0.195,0.577,0.423,9.68,-15.40,1.0,0.8,0.87,0.51,33.51,7.34,0.20,0.03,17.25,8.0,both,PTS,DraftKings Pick6,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-111.0,100.0,0.526,0.500,16.4,16.5,4.62,0.9,1.0,-0.195,0.577,0.423,9.68,-15.40,1.0,0.8,0.87,0.51,33.51,7.34,0.20,0.03,17.25,8.0


In [22]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY_x,LINE_BOOKMAKER_x,OPPONENT_x,TEAM_SPREAD_x,GAME_TOTAL_x,OPP_DEF_RATING_x,OPP_RANK_DEF_RATING_x,OPP_PACE_x,OPP_PACE_RANK_x,ODDS_OVER_x,ODDS_UNDER_x,IMP_PROB_OVER_x,IMP_PROB_UNDER_x,AVG_STAT_L10_x,MED_STAT_L10_x,STD_STAT_L10_x,EDGE_x,MED_EDGE_x,Z_SCORE_x,PROB_OVER_x,PROB_UNDER_x,EV_OVER_x,EV_UNDER_x,OVER_RATE_L5_x,OVER_RATE_L10_x,OVER_RATE_L15_x,OVER_RATE_SEASON_x,AVG_MIN_L10_x,STD_MIN_L10_x,AVG_USG_L10_x,STD_USG_L10_x,AVG_STAT_VS_MATCHUP_x,MATCHUP_GAMES_x,_merge,CATEGORY_y,LINE_BOOKMAKER_y,OPPONENT_y,TEAM_SPREAD_y,GAME_TOTAL_y,OPP_DEF_RATING_y,OPP_RANK_DEF_RATING_y,OPP_PACE_y,OPP_PACE_RANK_y,ODDS_OVER_y,ODDS_UNDER_y,IMP_PROB_OVER_y,IMP_PROB_UNDER_y,AVG_STAT_L10_y,MED_STAT_L10_y,STD_STAT_L10_y,EDGE_y,MED_EDGE_y,Z_SCORE_y,PROB_OVER_y,PROB_UNDER_y,EV_OVER_y,EV_UNDER_y,OVER_RATE_L5_y,OVER_RATE_L10_y,OVER_RATE_L15_y,OVER_RATE_SEASON_y,AVG_MIN_L10_y,STD_MIN_L10_y,AVG_USG_L10_y,STD_USG_L10_y,AVG_STAT_VS_MATCHUP_y,MATCHUP_GAMES_y
28,Quentin Grimes,PTS,6.5,18.05,24.14,31.36,3.35,10.48,23.00,0.672,0.328,player_points,Underdog,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-114.0,106.0,0.533,0.485,9.6,6.5,7.31,3.1,0.0,-0.424,0.664,0.336,24.65,-30.78,0.4,0.5,0.60,0.76,22.42,4.14,0.17,0.07,10.31,13.0,both,PTS,DraftKings Pick6,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-114.0,106.0,0.533,0.485,9.6,6.5,7.31,3.1,0.0,-0.424,0.664,0.336,24.65,-30.78,0.4,0.5,0.60,0.76,22.42,4.14,0.17,0.07,10.31,13.0
1,Jayson Tatum,AST,5.5,24.53,28.95,38.09,1.90,4.22,9.08,0.472,0.528,player_assists,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-132.0,110.0,0.569,0.476,6.9,7.0,2.56,1.4,1.5,-0.547,0.708,0.292,24.44,-38.68,0.6,0.7,0.67,0.53,36.19,4.56,0.27,0.04,6.80,10.0,both,AST,PrizePicks,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-132.0,110.0,0.569,0.476,6.9,7.0,2.56,1.4,1.5,-0.547,0.708,0.292,24.44,-38.68,0.6,0.7,0.67,0.53,36.19,4.56,0.27,0.04,6.80,10.0
23,Paul George,PTS,15.5,23.04,28.55,36.51,6.43,14.03,27.57,0.527,0.472,player_points,Underdog,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-111.0,100.0,0.526,0.500,16.4,16.5,4.62,0.9,1.0,-0.195,0.577,0.423,9.68,-15.40,1.0,0.8,0.87,0.51,33.51,7.34,0.20,0.03,17.25,8.0,both,PTS,PrizePicks,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-111.0,100.0,0.526,0.500,16.4,16.5,4.62,0.9,1.0,-0.195,0.577,0.423,9.68,-15.40,1.0,0.8,0.87,0.51,33.51,7.34,0.20,0.03,17.25,8.0
20,Jaylen Brown,PTS,26.5,28.78,37.53,42.25,11.90,23.97,40.61,0.487,0.513,player_points,Underdog,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,105.0,-114.0,0.488,0.533,25.7,25.5,5.83,-0.8,-1.0,0.137,0.446,0.554,-8.57,4.00,0.2,0.2,0.47,0.44,34.64,5.61,0.33,0.04,24.54,13.0,both,PTS,DraftKings Pick6,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,105.0,-114.0,0.488,0.533,25.7,25.5,5.83,-0.8,-1.0,0.137,0.446,0.554,-8.57,4.00,0.2,0.2,0.47,0.44,34.64,5.61,0.33,0.04,24.54,13.0
19,Tyrese Maxey,PTS,24.5,31.49,41.26,45.06,11.42,24.50,42.80,0.631,0.369,player_points,Underdog,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-104.0,-116.0,0.510,0.537,25.0,24.5,5.46,0.5,0.0,-0.092,0.537,0.463,5.33,-13.79,0.8,0.5,0.47,0.67,37.96,4.71,0.27,0.05,27.77,13.0,both,PTS,Underdog,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-104.0,-116.0,0.510,0.537,25.0,24.5,5.46,0.5,0.0,-0.092,0.537,0.463,5.33,-13.79,0.8,0.5,0.47,0.67,37.96,4.71,0.27,0.05,27.77,13.0


### Get top EVs for 2 legs

In [23]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 9  |  Pairs: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [24]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 10  |  Pairs: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [25]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 8  |  Pairs: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [26]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 9  |  Pairs: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [27]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 9  |  Triples: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [28]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 10  |  Triples: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [29]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 9  |  Triples: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [30]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 8  |  Triples: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
